# Practical 3A — Fashion-MNIST Initialization

## Aim
Study initialization on Fashion-MNIST using a deep multilayer perceptron.

## Setup
Run the installation cell if required. These notebooks use Python 3.10–3.12 where possible. On Apple Silicon, TensorFlow/PyTorch acceleration depends on the installed build; CPU execution is supported.

**Important:** dataset downloads and model training may take time. Results are generated when you run the notebook; no accuracy values are pre-filled.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn tensorflow

## Reproducibility\nA fixed random seed is used where practical. If a dataset download fails, check your internet connection and rerun the loading cell.

## 1. Load Fashion-MNIST and prepare splits

In [ ]:
import time, numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
np.random.seed(42); tf.random.set_seed(42)
(x_all,y_all),(x_test,y_test)=keras.datasets.fashion_mnist.load_data()
x_all=x_all.astype("float32")/255.; x_test=x_test.astype("float32")/255.
x_train,x_val=x_all[5000:],x_all[:5000]; y_train,y_val=y_all[5000:],y_all[:5000]
print(x_train.shape,x_val.shape,x_test.shape)

## 2. Model and gradient utilities

In [ ]:
def make_model(mode="poor"):
    init="random_normal" if mode=="poor" else ("glorot_uniform" if mode=="xavier" else "he_normal")
    model=keras.Sequential([layers.Input((28,28)),layers.Flatten()])
    for _ in range(5):
        model.add(layers.Dense(256,activation="relu",kernel_initializer=init))
    model.add(layers.Dense(10,activation="softmax"))
    model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return model

def gradient_norms(model, x, y):
    with tf.GradientTape() as tape:
        pred=model(x,training=True)
        loss=keras.losses.sparse_categorical_crossentropy(y,pred)
        loss=tf.reduce_mean(loss)
    grads=tape.gradient(loss,model.trainable_variables)
    return [float(tf.norm(g)) for g in grads if g is not None]

## 3. Compare poor, Xavier and He initialization

In [ ]:
models={}
gradient_records={}
for mode in ["poor","xavier","he"]:
    m=make_model(mode)
    gradient_records[mode]=gradient_norms(m,tf.convert_to_tensor(x_train[:128]),tf.convert_to_tensor(y_train[:128]))
    h=m.fit(x_train,y_train,validation_data=(x_val,y_val),epochs=20,batch_size=256,verbose=0)
    models[mode]=(m,h)
    print(mode,"final val accuracy",h.history["val_accuracy"][-1])

In [ ]:
plt.figure(figsize=(9,4))
for k,v in gradient_records.items(): plt.plot(v,label=k)
plt.yscale("log"); plt.title("Gradient norms by trainable tensor"); plt.xlabel("Trainable tensor index"); plt.ylabel("Gradient norm (log scale)"); plt.legend(); plt.show()
for k,(m,h) in models.items():
    plt.plot(h.history["val_loss"],label=k)
plt.title("Validation loss over 20 epochs"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.show()

**Conclusion:** Compare gradient magnitudes and validation curves. This is a ReLU network; small random initialization can reduce signal propagation. Xavier and He are standard variance-aware initializers (He is designed for ReLU-like activations).